In [0]:
from pyspark.sql.functions import *

In [0]:
events_df = spark.read.table("live_nation_prod.bronze_prod.events")
sales_df = spark.read.table("live_nation_prod.bronze_prod.ticket_sales")

In [0]:
events_df.columns

In [0]:
# From my Exploratort Notebook, I found out that there are different formats for the concert and a lot of add ones. This change cleans the name of the events.

events_df = events_df.withColumn(
    "event_name",
    when(
        col("event_name") == "New Harbor: Super Awesome Tour",
        "Neon Harbor - Super Awesome Tour"
    ).otherwise(col("event_name"))
)

In [0]:
# This statement creates a new field that categorizes a record as an event or an upsell. 
events_df = events_df.withColumn(
    "event_type",
    when(
        col("event_name") == 'Neon Harbor - Super Awesome Tour','event'
    ).otherwise('upsell')
)

In [0]:
display(events_df.limit(100))

In [0]:
# I created a temp view of a cleaned table to replicate the silver layer in medalion architecture. This is will make the SQL statement cleaner and more readable. 
events_df.createOrReplaceTempView("events_silver")

In [0]:
%sql
SELECT 
    e.event_id AS event_event_id,
    e.event_name AS event_event_name,
    e.event_type AS event_event_type,
    u.event_id AS upsell_event_id,
    u.event_name AS upsell_event_name,
    u.event_type AS upsell_event_type
FROM events_silver e
INNER JOIN events_silver u
    ON e.event_id = u.event_id
WHERE e.event_type = 'event'
  AND u.event_type = 'upsell'
